# RAG in 4 steps: chunking, indexing, retrieval, generation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hugobb/IndigenousAI/blob/main/notebooks/rag_workshop_beginner.ipynb)

RAG (Retrieval-Augmented Generation) lets an AI look things up before it answers. It cuts a document into chunks, indexes them, finds the chunks closest to your question, and writes its answer from them. Here the document is the programme of this workshop, which the AI has never read.

## Setup

**Do this:**
1. Before you run anything, make your own copy: click **File**, then **Save a copy in Drive**. Your copy opens in a new tab. Work in that tab, so your changes are saved.
2. In your copy, click ▶️ on the **Setup** cell below. If a message says "This notebook was not authored by Google", click **Run anyway**.
3. Wait until the cell shows **✅ Ready**. This takes 1 to 2 minutes.

In [ ]:
#@title Setup: click ▶️ and wait for ✅ Ready (1 to 2 minutes) { display-mode: "form" }
# You don't need to read this cell. It installs what is missing, loads the two AI models and
# defines the helpers the workshop uses. The interesting ones (ask, similarity, search, split,
# ask_with_documents) are also shown in each section, in a collapsed "Code" cell.
import logging
import warnings

# Hide three harmless notices: tqdm's progress bars and a macOS note from torch (outside Colab), and
# Colab's note that no Hugging Face token is set (the two models are public and need none).
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", message=r"\s*The secret `HF_TOKEN` does not exist in your Colab secrets")
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
try:
    import sentence_transformers  # noqa: F401
except ImportError:
    %pip install -q sentence-transformers transformers accelerate

import html
import os
import re
from collections.abc import Iterator
from dataclasses import dataclass
from queue import Empty
from threading import Thread
import numpy as np


# ======================================================================
# content.py
# ======================================================================
"""Everything the notebook works with: the programme, the excerpt drawn in step 1, and the question.

The document is the programme of this workshop (Future Imaginaries of Indigenous AI, Abundant
Intelligences, November 19 and 20, 2026), with its schedule written as prose and short speaker bios,
saved in documents/workshop_programme.txt. tests/test_effects.py (MiniLM) and tests/test_llm_effects.py
(the two Qwen models) check that the defaults below do what the notebook text says.
"""


def _load(name: str) -> str:
    """Read a file from documents/. When the notebook is built, each call is replaced by the text itself."""
    from pathlib import Path

    return (Path(__file__).resolve().parent / "documents" / name).read_text(encoding="utf-8")


# ---------------------------------------------------------------- the programme
_HEADER, _BODY = 'Title: Future Imaginaries of Indigenous AI, workshop programme (Abundant Intelligences)\nSource: https://abundant-intelligences.net/ka-hawaii-pae-aina-pod/\nRetrieved: 2026-11-20\nLicense: programme by the workshop organisers; lab, pod and speaker facts from abundant-intelligences.net (team, AIAI Lab and Ka Hawaiʻi Pae ʻĀina Pod pages), the speakers\' own pages, and Wikipedia "Solomon Enos" (oldid 1379642239, CC BY-SA 4.0)\nShortened: schedule rewritten as prose, speaker bios added.\n\nFuture Imaginaries of Indigenous AI is a two-day workshop by Abundant Intelligences, held on November 19 and 20, 2026, at the Create(x) Labs of the University of Hawaiʻi – West Oʻahu. It is hosted by the Ka Hawaiʻi Pae ʻĀina Pod. The workshop has two goals: to imagine machines of the future that fit the needs and hopes of participants\' communities, through a guided creation exercise, and to give a basic, hands-on introduction to machine learning.\n\n== About Abundant Intelligences ==\nAbundant Intelligences develops AI methods grounded in Indigenous epistemologies, aimed at abundance rather than scarcity. The program works along three axes. The Integrations Axis brings Indigenous knowledge practices together with the knowledge frameworks used by AI researchers. The Imaginaries Axis develops future imaginaries and speculative designs of Indigenous AI. The Intelligences Axis brings Indigenous perspectives to technical challenges in AI research. In short: we listen to each other, we dream our futures together, and we build those visions of the future.\n\n== The Future Imaginaries workshops ==\nThe Future Imaginaries of Indigenous AI workshops belong to the Imaginaries Axis. They help communities, practitioners and critics form their own vocabularies and visions of the futures they want, without falling into existing ideas of how these technologies should develop.\n\n== Day 1: Thursday, November 19 ==\nThe first day runs from 1:00 pm to 4:45 pm in room G-129 of Create(x). It opens at 1:00 pm with a 15-minute welcome: Kari Noe, director of the Create(x) Lab, introduces the Hawaiʻi Pod, and Ceyda Yolgörmez introduces the Future Imaginaries of Indigenous AI. At 1:15 pm, Solomon Enos and Kamuela Enos give a 30-minute talk, "Imagining abundant intelligences". After a 15-minute break, Suzanne Kite leads an hour-long imagining exercise at 2:00 pm, "Towards the future of sovereign intelligences". After a second 15-minute break, the imaginaries workshop continues at 3:15 pm for an hour and a half. The day closes at 4:45 pm.\n\n== Day 2: Friday, November 20 ==\nThe second day runs from 9:30 am to 4:00 pm in room D-141. It opens at 9:30 am with a 15-minute welcome. From 9:45 am to 10:30 am, the session "Setting the parameters of future AIs" covers existing philosophies, imaginaries and applications in the Ka Hawaiʻi Pae ʻĀina Pod, with talks by Manulani Meyer, Mele Moniz and Purple Maiʻa. After a 15-minute break, Karim Jerbi and Jason Lewis introduce the AIAI Lab at 10:45 am. Then Hugo Berard and Hamza Abdelhedi give the introduction to machine learning. Together the two sessions run 75 minutes, until noon. Lunch is from 12:00 to 1:00 pm. At 1:00 pm, participants start the 75-minute hands-on session, "Running your own AI". After a 15-minute break, Yorguin José Mantilla Ramos gives a 15-minute talk at 2:30 pm, "Can we trust AI?". The hands-on workshop then continues for 45 minutes, from 2:45 pm to 3:30 pm. At 3:30 pm, Jason Lewis and Kari Noe lead 30 minutes of reflections and future directions, and the workshop closes at 4:00 pm.\n\n== The Ka Hawaiʻi Pae ʻĀina Pod ==\nThe Ka Hawaiʻi Pae ʻĀina Pod is one of the research pods of Abundant Intelligences, based on Oʻahu. It is hosted by the Create(x) Immersive Media and Data Visualization Lab at the University of Hawaiʻi – West Oʻahu, and its community partner is Purple Maiʻa. Its work covers environmental stewardship, language revitalization, creative practice and community-based data stewardship. Kari Noe is the pod director and Anolani Saizon is the pod coordinator.\n\n== The AIAI Lab ==\nThe AIAI Lab is the technical team of Abundant Intelligences. It builds technical capacity for Indigenous-led AI projects: it gives technical advice, helps build prototypes, connects technical and non-technical collaborators, and runs training and events, including curriculum for the Summer Institute for Indigenous AI. The lab follows Indigenous data sovereignty principles, including OCAP and CARE. Its co-directors are Karim Jerbi of the Université de Montréal and Jason Edward Lewis of Concordia University. Hugo Berard is a postdoctoral researcher in the lab, and its student tech team includes Hamza Abdelhedi, Yorguin Mantilla Ramos and Antoine Bellemare. The lab works with Concordia University, the Université de Montréal, Mila, IVADO and UNIQUE.\n\n== Speakers ==\n=== Kari Noe ===\nKari Noe co-directs the Create(x) emerging media lab at the University of Hawaiʻi – West Oʻahu and is a PhD research assistant at the LAVA lab at the University of Hawaiʻi at Mānoa. She works on human-computer interaction, extended reality and game development, especially how emerging media supports learning.\n\n=== Ceyda Yolgörmez ===\nCeyda Yolgörmez is a postdoctoral researcher at Concordia University, at the Indigenous Futures Research Cluster of Abundant Intelligences. Her PhD combined social theory with interactive technologies such as large machine learning models and social robots, and proposes a "sociology of machines".\n\n=== Solomon Enos ===\nSolomon Enos is a Native Hawaiian artist, illustrator and activist, born in Mākaha Valley on Oʻahu. Solomon Enos\'s work has been shown at the National Museum of the American Indian, the Bishop Museum and the Pitt Rivers Museum in Oxford, England. Solomon Enos illustrated the books Akua Hawaiʻi and The Epic Tale of Hiʻiakaikapoliopele, and received a grant from the Joan Mitchell Foundation in 2017.\n\n=== Kamuela Enos ===\nKamuela Enos is a co-investigator of Abundant Intelligences at the University of Hawaiʻi at Mānoa and a member of the Ka Hawaiʻi Pae ʻĀina Pod. He founded the university\'s Office of Indigenous Innovation, which researches and scales traditional practices for Indigenous communities, and was director of social enterprise at MAʻO Organic Farms.\n\n=== Suzanne Kite ===\nSuzanne Kite is an Oglála Lakȟóta performance artist, composer and scholar. She directs the Wihanble S\'a Center for Indigenous AI at Bard College.\n\n=== Manulani Aluli Meyer ===\nManulani Aluli Meyer works on Indigenous epistemology. She holds a doctorate in philosophy of education from Harvard and is Konohiki for Kūlana o Kapolei at the University of Hawaiʻi – West Oʻahu. She wrote Hoʻoulu: Our Time of Becoming.\n\n=== Melemaikalani Moniz ===\nMelemaikalani (Mele) Moniz is a Kanaka ʻŌiwi legal scholar and a postdoctoral researcher at the University of Hawaiʻi at Mānoa. Her work covers Western legal systems, environmental justice and Indigenous rights. She holds an SJD in Indigenous Peoples Law and Policy from the University of Arizona.\n\n=== Jason Edward Lewis ===\nJason Edward Lewis is a digital media theorist, poet and software designer. He is Professor of Computation Arts at Concordia University, holds the University Research Chair in Computational Media and the Indigenous Future Imaginary, and founded the Obx Laboratory for Experimental Media.\n\n=== Karim Jerbi ===\nKarim Jerbi is Professor of Psychology at the Université de Montréal and a member of Mila. He holds a Canada Research Chair in Computational Neuroscience and directs UNIQUE, the Quebec Neuro-AI research centre. He studies brain oscillations and networks, and the meeting points of neuroscience, AI and art.\n\n=== Hugo Berard ===\nHugo Berard is a postdoctoral researcher with Abundant Intelligences, in the AIAI Lab. He holds a PhD in computer science from Mila, where he applied game theory to improve image-generation algorithms. He works on responsible AI built from local initiatives and community needs, and on how AI can support citizen participation in local decisions. He is active on Mila\'s sustainability committee, which works to reduce AI\'s environmental footprint, and facilitates Climate Fresk workshops.\n\n=== Hamza Abdelhedi ===\nHamza Abdelhedi is a PhD student in biomedical engineering at the Université de Montréal, in the CoCo Lab and at Mila, supervised by Karim Jerbi, and a member of the AIAI Lab. He holds an MSc in computer science (artificial intelligence) from the Université de Montréal and an engineering diploma in telecommunications from SUP\'COM in Tunisia. He uses M/EEG recordings and computational models to study brain dynamics and to compare neural networks with human brain activity, and maintains open-source tools such as MNE-Denoise.\n\n=== Yorguin José Mantilla Ramos ===\nYorguin José Mantilla Ramos is an electronics engineer and a member of the AIAI Lab\'s student tech team. He is interested in the interpretability and explainability of models, how data and knowledge are represented, how cognition works, and the philosophy of science.\n\n=== Purple Maiʻa ===\nPurple Maiʻa is a nonprofit based in Honolulu. It teaches and supports culturally grounded technology makers, mainly Native Hawaiian and Pacific Islander learners.\n'.split("\n\n", 1)
SOURCE = dict(line.split(": ", 1) for line in _HEADER.splitlines())  # Title, Source, Retrieved, License, Shortened
PROGRAMME = _BODY.strip()  # the programme without its 5-line attribution block
PROGRAMME_URL = SOURCE["Source"]
PROGRAMME_RETRIEVED = SOURCE["Retrieved"]

# ---------------------------------------------------------------- 1. Chunking
# The part of the programme drawn in step 1: the start of the Day 2 schedule, which holds the answer.
_DAY2 = PROGRAMME.split("== Day 2: Friday, November 20 ==\n", 1)[1]
EXCERPT = _DAY2[: _DAY2.index("until noon.") + len("until noon.")]
SIZE = 60  # words per chunk, the default in step 1
OVERLAP = 10
TRY_SIZE = 80  # step 2 suggests it: the chunk with the answer drops out of the top 3

# ---------------------------------------------------------------- 2. Retrieval and 3. Generation
QUESTION = "Who gives the introduction to machine learning?"
ANSWER = "Hugo Berard and Hamza Abdelhedi"  # in the best chunk at SIZE and OVERLAP
NAME = "Berard"  # in the AI's answer with the chunks, not without them


# ======================================================================
# toolkit.py
# ======================================================================
"""The workshop's machinery: chunking, similarity, search, prompts and the two AI models."""



EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"


class MiniLMEmbedder:
    """Turns text into embeddings (384 numbers each) with a small, fast model."""

    def __init__(self, device: str):
        from sentence_transformers import SentenceTransformer

        self.model = SentenceTransformer(EMBED_MODEL, device=device)

    def embed(self, texts: list[str]) -> np.ndarray:
        return self.model.encode(list(texts), normalize_embeddings=True, convert_to_numpy=True)


# ---------------------------------------------------------------- the answer-writing model
CHAT_MODEL_GPU = "Qwen/Qwen3-1.7B"
CHAT_MODEL_CPU = "Qwen/Qwen3-0.6B"
MAX_NEW_TOKENS = 200


def model_dtype(device: str, model_name: str):
    """Full precision (float32) on the GPU, where half precision (float16) can overflow with Qwen3 on a T4,
    and for the small CPU model, because bfloat16 can be very slow on CPUs like Colab's. Larger models on
    the CPU use bfloat16, half the memory."""
    import torch

    return torch.float32 if device == "cuda" or model_name == CHAT_MODEL_CPU else torch.bfloat16


def dtype_argument(device: str, model_name: str) -> dict:
    """The precision argument for from_pretrained. transformers 4.56 renamed torch_dtype to dtype:
    older versions crash on dtype=, newer ones print a warning for torch_dtype=."""
    import transformers

    major, minor = (int(part) for part in transformers.__version__.split(".")[:2])
    return {"dtype" if (major, minor) >= (4, 56) else "torch_dtype": model_dtype(device, model_name)}


def stream_worker(target, streamer) -> Iterator[str]:
    """Run `target` (the model writing) in the background and pass on its text as it arrives.
    If it fails, stop waiting and raise instead of hanging forever."""
    failure: list[BaseException] = []

    def work():
        try:
            target()
        except BaseException as error:  # noqa: BLE001 - any failure must reach the participant
            failure.append(error)
            streamer.text_queue.put(streamer.stop_signal)

    worker = Thread(target=work, daemon=True)
    worker.start()
    try:
        yield from streamer
    except Empty:
        raise RuntimeError("The model took too long to answer.") from None
    worker.join(timeout=1)
    if failure:
        raise RuntimeError(f"The model stopped with an error: {failure[0]}") from failure[0]


class QwenGenerator:
    """Writes answers with a small Qwen model. Greedy decoding: the same question gives the same answer."""

    def __init__(self, model_name: str, device: str):
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name, **dtype_argument(device, model_name)).to(device)
        self.model.eval()

    def generate(self, prompt: str) -> Iterator[str]:
        from transformers import TextIteratorStreamer

        text = self.tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
        streamer = TextIteratorStreamer(self.tokenizer, skip_prompt=True, skip_special_tokens=True, timeout=180)
        settings = dict(
            **inputs,
            streamer=streamer,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
        )
        yield from stream_worker(lambda: self.model.generate(**settings), streamer)


class FakeGenerator:
    """A stand-in for tests: repeats the first passage instead of thinking."""

    def generate(self, prompt: str) -> Iterator[str]:
        yield "(test answer) "
        first = next((line for line in prompt.splitlines() if line.startswith("[1] ")), prompt)
        for word in first.removeprefix("[1] ").split():
            yield word + " "


@dataclass
class Models:
    embedder: MiniLMEmbedder
    generator: QwenGenerator | FakeGenerator
    device: str
    llm_name: str


MODELS: Models | None = None
NOT_READY = "Run the Setup cell first."


def load_models() -> Models:
    """Load both models, on the GPU when there is one. The result is also kept in MODELS."""
    global MODELS
    MODELS = _load_models()
    return MODELS


def _load_models() -> Models:
    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    embedder = MiniLMEmbedder(device)
    if os.environ.get("RAG_WORKSHOP_FAKE_LLM") == "1":
        return Models(embedder, FakeGenerator(), device, "fake")
    name = os.environ.get("RAG_WORKSHOP_LLM") or (CHAT_MODEL_GPU if device == "cuda" else CHAT_MODEL_CPU)
    return Models(embedder, QwenGenerator(name, device), device, name)


def status_line(device: str) -> str:
    """The Setup cell's last line. Without a GPU (the default in Colab), answers are slower."""
    ready = "✅ Ready. You can go on to section 1."
    return ready if device == "cuda" else ready + " Each answer takes up to a minute."


# ---------------------------------------------------------------- plain helpers for the notebook cells
SHOW = 60  # words of a passage shown on screen: a whole chunk of up to 60 words, the start of a longer one


def _short(text: str) -> str:
    words = text.split()
    return " ".join(words[:SHOW]) + (" ..." if len(words) > SHOW else "")


def ready() -> bool:
    """True once the Setup cell has loaded the models; otherwise says to run it."""
    if MODELS is None:
        print(NOT_READY)
        return False
    return True


def write_answer(prompt: str) -> None:
    """Send the prompt to the AI model and print its answer as it is written."""
    for piece in MODELS.generator.generate(prompt):
        print(piece, end="", flush=True)
    print()


def show_passages(passages: list[str]) -> None:
    print("Passages sent to the AI:")
    for n, passage in enumerate(passages, start=1):
        print(f"{n}. {_short(passage)}")
    print()


COLOURS = ["#a8cbff", "#ffc979", "#a9e3a9", "#eeb0e8"]  # blue, orange, green, pink: light enough for black text


def show_chunking(text: str, size: int, overlap: int) -> None:
    """Draw how `text` is cut: each chunk is one coloured block; where two chunks overlap, the words get
    both colours, in stripes."""
    chunks = chunk_overlap(text, size, overlap) if size > 0 else [text]
    words = text.split()
    overlap = max(0, min(overlap, size - 1)) if size > 0 else 0
    step = size - overlap if size > 0 else len(words)
    # Group the words into runs: a run is in one chunk only, or in two (an overlap).
    runs: list[tuple[int, bool, list[str]]] = []  # (chunk, in the chunk before too?, words)
    for i, word in enumerate(words):
        n = min(i // step, len(chunks) - 1)  # the last chunk that holds this word
        shared = n > 0 and i < (n - 1) * step + size
        if runs and runs[-1][:2] == (n, shared):
            runs[-1][2].append(word)
        else:
            runs.append((n, shared, [word]))
    colour = lambda n: COLOURS[n % len(COLOURS)]  # noqa: E731
    blocks = []
    for k, (n, shared, run) in enumerate(runs):
        if shared:
            a, b = colour(n - 1), colour(n)
            fill = f"repeating-linear-gradient(135deg, {a} 0 7px, {b} 7px 14px)"  # stripes of both colours
        else:
            fill = colour(n)
        label = f'<b style="color:#555;font-size:75%">[{n + 1}] </b>' if k == 0 or runs[k - 1][0] != n else ""  # where chunk n starts
        end = " " if k + 1 < len(runs) else ""  # the space inside the block, so the colour has no gaps
        style = f"background:{fill};color:#000;padding:3px 0;box-decoration-break:clone;-webkit-box-decoration-break:clone"
        blocks.append(f'<span style="{style}">{label}{html.escape(" ".join(run))}{end}</span>')
    head = f"Cut into {len(chunks)} {'chunk' if len(chunks) == 1 else 'chunks'} of {size} words, overlap {overlap}."
    note = " Striped words are in two chunks." if overlap else ""
    body = f'<p>{head}{note}</p><p style="line-height:2.2">{"".join(blocks)}</p>'
    try:
        from IPython.display import HTML, display

        display(HTML(body))
    except ImportError:
        print(head)
        for n, chunk in enumerate(chunks, start=1):
            print(f"{n}. {chunk}")


@dataclass
class Index:
    """The chunks and their embeddings, computed once (step 2), so search does not recompute them."""

    chunks: list[str]
    vectors: np.ndarray


MAP_COLOURS = ["#1f6fd1", "#e07b00", "#2a9d3a"]  # the 3 closest chunks: blue, orange, green


def _project_2d(vectors: np.ndarray, extra: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Squash the chunk embeddings to 2 numbers each (PCA), and put `extra` (the question) on the same map."""
    mean = vectors.mean(axis=0)
    centred = vectors - mean
    if len(vectors) < 2:
        return np.zeros((len(vectors), 2)), np.zeros(2)
    _, _, axes = np.linalg.svd(centred, full_matrices=False)
    axes = axes[:2]
    if len(axes) < 2:
        axes = np.vstack([axes, np.zeros_like(axes[0])])
    return centred @ axes.T, (extra - mean) @ axes.T


def draw_map(index: Index, question: str, best: list[tuple[float, str]]) -> None:
    """Show the map of the chunks (see map_png) in the notebook."""
    png = map_png(index, question, best)
    try:
        from IPython.display import Image, display

        display(Image(data=png))
    except ImportError:
        print("(map drawn: run in a notebook to see it)")


def map_png(index: Index, question: str, best: list[tuple[float, str]]) -> bytes:
    """The chunks as dots, the question as a star, and lines to the closest chunks, as a PNG image."""
    import io

    from matplotlib.figure import Figure

    points, star = _project_2d(np.asarray(index.vectors, dtype=float), np.asarray(embed([question])[0], dtype=float))
    fig = Figure(figsize=(7.5, 5.2), dpi=110)
    ax = fig.subplots()
    ax.scatter(points[:, 0], points[:, 1], s=60, color="#c8c8c8", edgecolor="#888", zorder=2)
    for n, (x, y) in enumerate(points, start=1):
        ax.annotate(str(n), (x, y), textcoords="offset points", xytext=(5, 4), fontsize=8, color="#666")
    for rank, (score, chunk) in enumerate(best):
        n = index.chunks.index(chunk)
        x, y = points[n]
        colour = MAP_COLOURS[rank % len(MAP_COLOURS)]
        ax.plot([star[0], x], [star[1], y], color=colour, linewidth=1.5, zorder=1)
        ax.scatter([x], [y], s=140, color=colour, edgecolor="black", zorder=3, label=f"#{rank + 1}: chunk {n + 1} ({score:.2f})")
    ax.scatter([star[0]], [star[1]], s=380, marker="*", color="#d62728", edgecolor="black", zorder=4, label="your sentence")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"The {len(index.chunks)} chunks on a map: close dots have close meanings", fontsize=11)
    ax.legend(loc="best", fontsize=8, framealpha=0.9)
    fig.tight_layout()
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png")
    return buffer.getvalue()


def _bare(text: str) -> list[str]:
    """The words of a text, without the punctuation around them ("foot." and "foot" match)."""
    return [word.strip(".,;:!?\"'()“”‘’") for word in text.split()]


def where_is(chunks: list[str], phrase: str) -> None:
    """Say which chunks contain a phrase, or show where it was cut in two."""
    norm = " ".join(phrase.split()).lower()
    flat = [" ".join(c.split()).lower() for c in chunks]
    found = [i + 1 for i, c in enumerate(flat) if norm and norm in c]
    if found:
        label = "chunk" if len(found) == 1 else "chunks"
        print(f'"{phrase}" is in {label} {", ".join(map(str, found))}.')
        return
    words = _bare(norm)
    for i in range(len(flat) - 1):
        left, right = _bare(flat[i]), _bare(flat[i + 1])
        for j in range(len(words) - 1, 0, -1):
            if left[-j:] == words[:j] and right[: len(words) - j] == words[j:]:
                print(f'"{phrase}" was cut in two, between chunk {i + 1} and chunk {i + 2}.')
                print(f"Chunk {i + 1} ends with: ...{chunks[i][-80:]}")
                print(f"Chunk {i + 2} starts with: {chunks[i + 1][:80]}...")
                return
    print(f'"{phrase}" is not in any chunk.')


_ATTRIBUTION = ("Title:", "Source:", "Retrieved:", "License:", "Shortened:")
_HEADING = re.compile(r"^(={2,3})\s*(.+?)\s*\1\s*$")


def sections(document: str) -> list[str]:
    """One passage per section (== and === headings). The heading path is the first line."""
    lines = document.splitlines()
    head = []
    for line in lines:
        if not line.strip():
            break
        head.append(line)
    if head and all(line.startswith(_ATTRIBUTION) for line in head):
        lines = lines[len(head) :]

    passages: list[str] = []
    title = "Introduction"
    parent = ""
    body: list[str] = []

    def flush() -> None:
        text = "\n".join(body).strip()
        if text:
            passages.append(f"{title}\n{text}")
        body.clear()

    for line in lines:
        match = _HEADING.match(line.strip())
        if not match:
            body.append(line)
            continue
        flush()
        name = match.group(2)
        if len(match.group(1)) == 2:
            parent = name
            title = name
        else:
            title = f"{parent}: {name}" if parent else name
    flush()
    return passages


# ---------------------------------------------------------------- shown in the notebook, one section at a time
# Each function below is also shown, as it is here, in a collapsed "Code" cell of its section.

# Step 4 (with build_prompt and ask_with_documents below)
def ask(question: str) -> None:
    """Ask the AI model alone: no documents, only what it learned during its training."""
    if not ready():
        return
    print("Answer:")
    write_answer(question)  # the question is sent as it is, with nothing added


# Step 2
def embed(texts: list[str]) -> np.ndarray:
    """Turn each text into an embedding: a list of 384 numbers that captures its meaning."""
    return MODELS.embedder.embed(texts)


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    """How close two embeddings point: 1 = same direction (same meaning), 0 = unrelated."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def score(a: str, b: str) -> float:
    """The similarity score of two texts: embed both, then compare the two lists of numbers."""
    if MODELS is None:
        print(NOT_READY)
        return 0.0
    numbers_a, numbers_b = embed([a, b])
    return cosine(numbers_a, numbers_b)


def similarity(a: str, b: str) -> None:
    """Print how similar two texts are, e.g. 'Similarity: 0.83'."""
    if not ready():
        return
    print(f"Similarity: {score(a, b):.2f}")


def build_index(chunks: list[str]) -> Index | None:
    """Indexing: compute the embedding of every chunk once, and keep it next to its chunk."""
    if not ready():
        return None
    index = Index(list(chunks), embed(list(chunks)))
    print(f"Indexed {len(index.chunks)} chunks: one list of {index.vectors.shape[1]} numbers per chunk.")
    return index


def show_map(index: Index, sentence: str, top: int = 3) -> None:
    """Draw every chunk on a map, put the sentence on it, and list the `top` closest chunks."""
    if not ready() or index is None:
        return
    best = top_passages(sentence, index, top)
    draw_map(index, sentence, best)
    for rank, (score, chunk) in enumerate(best, start=1):
        print(f"#{rank}  chunk {index.chunks.index(chunk) + 1}  score {score:.2f}  {_short(chunk)}")


# Step 3
def top_passages(question: str, index: Index | list[str], top: int = 3) -> list[tuple[float, str]]:
    """The `top` chunks closest in meaning to the question, best first, as (score, chunk)."""
    if not isinstance(index, Index):  # plain chunks, not indexed yet: embed them now
        index = Index(list(index), embed(list(index)) if index else np.zeros((0, 384)))
    if not index.chunks:
        return []
    # 1. Embed the question (the chunks were embedded once, when the index was built).
    question_numbers = embed([question])[0]
    # 2. Score every chunk against the question.
    scored = [(cosine(question_numbers, numbers), chunk) for numbers, chunk in zip(index.vectors, index.chunks)]
    # 3. Sort from the highest score to the lowest, and keep the first `top`.
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return scored[:top]


def search(question: str, passages: Index | list[str], top: int = 3) -> None:
    """Print the passages closest to the question: rank, score, start of the text."""
    if not ready():
        return
    for n, (s, passage) in enumerate(top_passages(question, passages, top), start=1):
        print(f"{n}. {s:.2f}  {_short(passage)}")


# Step 1
def chunk_overlap(text: str, size: int, overlap: int) -> list[str]:
    """Cut text into pieces of `size` words; each piece repeats the last `overlap` words of the one before."""
    overlap = max(0, min(overlap, size - 1))  # an overlap as long as the chunk would never move forward
    step = size - overlap  # how far each new chunk starts after the previous one
    words = text.split()
    chunks = []
    for start in range(0, len(words), step):
        chunks.append(" ".join(words[start : start + size]))
        if start + size >= len(words):  # this chunk reached the last word
            break
    return chunks


def split(document: str, size: int = 60, overlap: int = 0) -> list[str]:
    """Cut a document into chunks of `size` words. 0 words means one chunk for everything."""
    if size <= 0:
        chunks = [document.strip()] if document.strip() else []
    else:
        chunks = chunk_overlap(document, size, max(0, overlap))
    print(f"Cut into {len(chunks)} {'chunk' if len(chunks) == 1 else 'chunks'}.")
    return chunks


# Step 4
def build_prompt(question: str, contexts: list[str], allow_unknown: bool) -> str:
    """The exact text sent to the model: instructions, the retrieved passages, the question."""
    lines = ["Answer the question using the context below."]
    if allow_unknown:
        lines.append('If the answer is not in the context, say "I don\'t know based on these documents."')
    lines += ["", "Context:"]
    lines += [f"[{i}] {c}" for i, c in enumerate(contexts, start=1)]
    lines += ["", f"Question: {question}", "Answer:"]
    return "\n".join(lines)


def ask_with_documents(question: str, passages: Index | list[str], top: int = 3, say_dont_know: bool = True) -> None:
    """RAG: find the passages closest to the question, then let the AI answer from them."""
    if not ready():
        return
    # 1. Retrieve: embed the question and every passage, keep the `top` closest (section 3).
    best = [passage for _, passage in top_passages(question, passages, top)]
    show_passages(best)
    # 2. Augment: paste those passages into the prompt, above the question.
    prompt = build_prompt(question, best, say_dont_know)
    # 3. Generate: the AI writes its answer from that prompt.
    print("Answer:")
    write_answer(prompt)


print("Loading the two AI models. This takes 1 to 2 minutes...")
MODELS = load_models()
print(status_line(MODELS.device))


## 1. Chunking

The programme is cut into chunks of a few words, and each chunk is searched on its own. `size` is the number of words per chunk. `overlap` is how many words each chunk repeats from the chunk before, so a sentence cut at the edge is still whole in one chunk.

**Do this:**
1. Run the cell. The coloured text is the start of the Day 2 schedule, cut with these settings: one colour per chunk, numbered [1], [2], .... Striped words are in two chunks: they are repeated.
2. Change `size` and `overlap`, and run the cell again.

In [ ]:
size = 60
overlap = 10
show_chunking(EXCERPT, size, overlap)
chunks = split(PROGRAMME, size, overlap)

In [ ]:
#@title 🔍 Code: how split() works (optional: double-click to see the code) { display-mode: "form" }
# The code behind chunk_overlap(), split(), exactly as the Setup cell defines it.
# You don't need to run this cell: Setup already did. Running it again changes nothing.


def chunk_overlap(text: str, size: int, overlap: int) -> list[str]:
    """Cut text into pieces of `size` words; each piece repeats the last `overlap` words of the one before."""
    overlap = max(0, min(overlap, size - 1))  # an overlap as long as the chunk would never move forward
    step = size - overlap  # how far each new chunk starts after the previous one
    words = text.split()
    chunks = []
    for start in range(0, len(words), step):
        chunks.append(" ".join(words[start : start + size]))
        if start + size >= len(words):  # this chunk reached the last word
            break
    return chunks


def split(document: str, size: int = 60, overlap: int = 0) -> list[str]:
    """Cut a document into chunks of `size` words. 0 words means one chunk for everything."""
    if size <= 0:
        chunks = [document.strip()] if document.strip() else []
    else:
        chunks = chunk_overlap(document, size, max(0, overlap))
    print(f"Cut into {len(chunks)} {'chunk' if len(chunks) == 1 else 'chunks'}.")
    return chunks


## 2. Indexing

Indexing computes the embedding of every chunk once, and keeps it next to its chunk. An embedding is a list of 384 numbers that captures what a chunk means. On the map, each dot is a chunk: dots close together have close meanings. The map squeezes 384 numbers into 2, so distances on it are only a picture; the scores are exact.

**Do this:**
1. Run the cell. The red star is the sentence, and the lines go to the 3 closest chunks, listed under the map.
2. Change the sentence to a question or sentence of your own, and run the cell again.

In [ ]:
index = build_index(chunks)
show_map(index, "Who gives the introduction to machine learning?")

In [ ]:
#@title 🔍 Code: how build_index() and show_map() work (optional: double-click to see the code) { display-mode: "form" }
# The code behind embed(), build_index(), show_map(), exactly as the Setup cell defines it.
# You don't need to run this cell: Setup already did. Running it again changes nothing.


def embed(texts: list[str]) -> np.ndarray:
    """Turn each text into an embedding: a list of 384 numbers that captures its meaning."""
    return MODELS.embedder.embed(texts)


def build_index(chunks: list[str]) -> Index | None:
    """Indexing: compute the embedding of every chunk once, and keep it next to its chunk."""
    if not ready():
        return None
    index = Index(list(chunks), embed(list(chunks)))
    print(f"Indexed {len(index.chunks)} chunks: one list of {index.vectors.shape[1]} numbers per chunk.")
    return index


def show_map(index: Index, sentence: str, top: int = 3) -> None:
    """Draw every chunk on a map, put the sentence on it, and list the `top` closest chunks."""
    if not ready() or index is None:
        return
    best = top_passages(sentence, index, top)
    draw_map(index, sentence, best)
    for rank, (score, chunk) in enumerate(best, start=1):
        print(f"#{rank}  chunk {index.chunks.index(chunk) + 1}  score {score:.2f}  {_short(chunk)}")


## 3. Retrieval

Search turns the question into an embedding, scores it against every chunk in the index, and keeps the closest chunks. The score goes from 0 (unrelated) to 1 (same meaning).

**Do this:**
1. Run the cell. Is the chunk with the answer, Hugo Berard and Hamza Abdelhedi, first?
2. Go back to step 1, set `size` to 80, and run steps 1, 2 and 3 again. Where is the answer now?

In [ ]:
question = "Who gives the introduction to machine learning?"
search(question, index, top=3)

In [ ]:
#@title 🔍 Code: how search() works (optional: double-click to see the code) { display-mode: "form" }
# The code behind cosine(), top_passages(), search(), exactly as the Setup cell defines it.
# You don't need to run this cell: Setup already did. Running it again changes nothing.


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    """How close two embeddings point: 1 = same direction (same meaning), 0 = unrelated."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def top_passages(question: str, index: Index | list[str], top: int = 3) -> list[tuple[float, str]]:
    """The `top` chunks closest in meaning to the question, best first, as (score, chunk)."""
    if not isinstance(index, Index):  # plain chunks, not indexed yet: embed them now
        index = Index(list(index), embed(list(index)) if index else np.zeros((0, 384)))
    if not index.chunks:
        return []
    # 1. Embed the question (the chunks were embedded once, when the index was built).
    question_numbers = embed([question])[0]
    # 2. Score every chunk against the question.
    scored = [(cosine(question_numbers, numbers), chunk) for numbers, chunk in zip(index.vectors, index.chunks)]
    # 3. Sort from the highest score to the lowest, and keep the first `top`.
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return scored[:top]


def search(question: str, passages: Index | list[str], top: int = 3) -> None:
    """Print the passages closest to the question: rank, score, start of the text."""
    if not ready():
        return
    for n, (s, passage) in enumerate(top_passages(question, passages, top), start=1):
        print(f"{n}. {s:.2f}  {_short(passage)}")


## 4. Generation

The AI writes the answer: first alone, then with the 3 chunks that search found pasted above the question.

**Do this:**
1. Run the cell and compare the two answers.
2. Change the question in step 3, or the chunking in step 1, and run the cells again from there.

In [ ]:
ask(question)
ask_with_documents(question, index, top=3)

In [ ]:
#@title 🔍 Code: how ask() and ask_with_documents() work (optional: double-click to see the code) { display-mode: "form" }
# The code behind ask(), build_prompt(), ask_with_documents(), exactly as the Setup cell defines it.
# You don't need to run this cell: Setup already did. Running it again changes nothing.


def ask(question: str) -> None:
    """Ask the AI model alone: no documents, only what it learned during its training."""
    if not ready():
        return
    print("Answer:")
    write_answer(question)  # the question is sent as it is, with nothing added


def build_prompt(question: str, contexts: list[str], allow_unknown: bool) -> str:
    """The exact text sent to the model: instructions, the retrieved passages, the question."""
    lines = ["Answer the question using the context below."]
    if allow_unknown:
        lines.append('If the answer is not in the context, say "I don\'t know based on these documents."')
    lines += ["", "Context:"]
    lines += [f"[{i}] {c}" for i, c in enumerate(contexts, start=1)]
    lines += ["", f"Question: {question}", "Answer:"]
    return "\n".join(lines)


def ask_with_documents(question: str, passages: Index | list[str], top: int = 3, say_dont_know: bool = True) -> None:
    """RAG: find the passages closest to the question, then let the AI answer from them."""
    if not ready():
        return
    # 1. Retrieve: embed the question and every passage, keep the `top` closest (section 3).
    best = [passage for _, passage in top_passages(question, passages, top)]
    show_passages(best)
    # 2. Augment: paste those passages into the prompt, above the question.
    prompt = build_prompt(question, best, say_dont_know)
    # 3. Generate: the AI writes its answer from that prompt.
    print("Answer:")
    write_answer(prompt)


Sources: the programme of Future Imaginaries of Indigenous AI (Abundant Intelligences), with facts from [abundant-intelligences.net](https://abundant-intelligences.net/ka-hawaii-pae-aina-pod/) and the speakers' own pages, retrieved 2026-11-20. The bio of Solomon Enos is from Wikipedia ([CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/)). Models: all-MiniLM-L6-v2 for the embeddings, Qwen3-0.6B (or Qwen3-1.7B with a GPU) for the answers.